# ============================================
# 10_industrial_park_load_downloader.ipynb
# Industrial Park Electric Power Load Dataset
# ============================================

## Dataset Title
Industrial Park Electric Power Load Dataset

## Source URL
Official repository page:  
https://github.com/Industrialpark/SEMLab_HFUT-Building-Electricpowerloaddata

Stable direct ZIP download URL:  
https://codeload.github.com/Industrialpark/SEMLab_HFUT-Building-Electricpowerloaddata/zip/refs/heads/main

## Purpose of This Dataset
This dataset is used as the real industrial-park load dataset in the PhD workflow.

It is especially useful for:
- industrial load forecasting
- multi-resolution time-series experiments
- optimization input generation
- industrial demand analysis
- deployment-oriented testing

## Why We Need This Dataset
We need this dataset because it provides real electric power load data from an industrial park setting rather than only household or building benchmarks.

According to the public repository:
- the data come from an industrial park in Suzhou, China
- the dataset covers 4 buildings
- it spans 6 years
- it includes 5 min, 30 min, and 1 hour resolutions
- the data are public

This makes it highly suitable for:
- Notebook 3: NEO-Forecast
- Notebook 6: NEO-Opt
- Notebook 7: NEO-Robust
- Notebook 10: industrial deployment-oriented data integration
- Notebook 11: Full System Simulation

## What We Are Doing in This Notebook
In this notebook, we will:

1. Mount Google Drive
2. Create dataset folders
3. Download the dataset ZIP from the stable GitHub codeload URL
4. Extract the ZIP
5. Identify usable files
6. Save a dataset manifest
7. Create a clean ZIP bundle
8. Save everything in Google Drive
9. Create a final package ZIP
10. Download that final package to the local PC

## Where This Data Will Be Used Next
This dataset will be used in:

- Notebook 3: NEO-Forecast
- Notebook 6: NEO-Opt
- Notebook 7: NEO-Robust
- Notebook 11: Full System Simulation

## Notes
The public repository README states that the dataset contains industrial park building electric-power load data from Suzhou, China, with 4 buildings, 6 years of data, and 5-minute, 30-minute, and 1-hour resolutions.

In [1]:
# ============================================
# 10_industrial_park_load_downloader.ipynb
# FULL FIXED SINGLE CELL
# ============================================

import os
import json
import shutil
import zipfile
from pathlib import Path
import requests
from google.colab import drive, files

# -----------------------------
# 1. Dataset metadata
# -----------------------------
DATASET_NAME = "Industrial Park Electric Power Load Dataset"
SOURCE_PAGE = "https://github.com/Industrialpark/SEMLab_HFUT-Building-Electricpowerloaddata"
DOWNLOAD_URL = "https://codeload.github.com/Industrialpark/SEMLab_HFUT-Building-Electricpowerloaddata/zip/refs/heads/main"

print("Dataset Name :", DATASET_NAME)
print("Source Page  :", SOURCE_PAGE)
print("Download URL :", DOWNLOAD_URL)

# -----------------------------
# 2. Mount Google Drive
# -----------------------------
drive.mount('/content/drive')

# -----------------------------
# 3. Create folder structure
# -----------------------------
project_root = Path("/content/drive/MyDrive/PhD_Thesis_Datasets")
dataset_root = project_root / "10_industrial_park_load"
raw_dir = dataset_root / "raw"
extracted_dir = dataset_root / "extracted"
bundle_dir = dataset_root / "bundle"

for folder in [project_root, dataset_root, raw_dir, extracted_dir, bundle_dir]:
    folder.mkdir(parents=True, exist_ok=True)

print("\nCreated folders:")
print(" -", project_root)
print(" -", dataset_root)
print(" -", raw_dir)
print(" -", extracted_dir)
print(" -", bundle_dir)

# -----------------------------
# 4. Robust download helper
# -----------------------------
def download_file(url: str, dest_path: Path, chunk_size: int = 1024 * 1024):
    headers = {"User-Agent": "Mozilla/5.0"}
    with requests.get(url, headers=headers, stream=True, timeout=600, allow_redirects=True) as r:
        r.raise_for_status()
        with open(dest_path, "wb") as f:
            for chunk in r.iter_content(chunk_size=chunk_size):
                if chunk:
                    f.write(chunk)
    return dest_path

# -----------------------------
# 5. Download dataset ZIP
# -----------------------------
download_path = raw_dir / "industrial_park_load_main.zip"

if not download_path.exists():
    print("\nDownloading dataset ZIP...")
    download_file(DOWNLOAD_URL, download_path)
    print("Download complete:", download_path)
else:
    print("\nUsing cached file:", download_path)

if not download_path.exists():
    raise FileNotFoundError(f"Dataset ZIP not found: {download_path}")

print("ZIP size:", round(download_path.stat().st_size / (1024 * 1024), 2), "MB")

# -----------------------------
# 6. Extract ZIP
# -----------------------------
print("\nExtracting ZIP...")
with zipfile.ZipFile(download_path, "r") as zf:
    zf.extractall(extracted_dir)

print("Extraction complete:", extracted_dir)

# -----------------------------
# 7. Collect usable files
# -----------------------------
all_files = sorted([p for p in extracted_dir.rglob("*") if p.is_file()])

usable_files = []
for p in all_files:
    rel = str(p.relative_to(extracted_dir))
    if "__MACOSX" in rel or p.name.startswith("._"):
        continue
    if p.suffix.lower() in [".csv", ".txt", ".json", ".xlsx", ".xls", ".md"]:
        usable_files.append(p)

print("\nTotal extracted files:", len(all_files))
print("Usable files:", len(usable_files))
for fp in usable_files[:40]:
    print(" -", fp)

# -----------------------------
# 8. Save manifest
# -----------------------------
manifest = {
    "dataset_name": DATASET_NAME,
    "source_page": SOURCE_PAGE,
    "download_url": DOWNLOAD_URL,
    "downloaded_file": str(download_path),
    "extracted_dir": str(extracted_dir),
    "all_files": [str(p) for p in all_files],
    "usable_files": [str(p) for p in usable_files],
    "used_in_next_notebooks": [
        "Notebook 3: NEO-Forecast",
        "Notebook 6: NEO-Opt",
        "Notebook 7: NEO-Robust",
        "Notebook 11: Full System Simulation"
    ]
}

manifest_path = dataset_root / "dataset_manifest.json"
with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2)

print("\nSaved manifest:", manifest_path)

# -----------------------------
# 9. Create clean bundle ZIP
# -----------------------------
clean_bundle_zip_path = bundle_dir / "industrial_park_load_CLEAN_bundle.zip"

with zipfile.ZipFile(clean_bundle_zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for fp in usable_files:
        if not fp.exists():
            continue
        arcname = str(fp.relative_to(extracted_dir))
        zf.write(fp, arcname=arcname)

print("Created clean bundle ZIP:", clean_bundle_zip_path)
print("Clean bundle size:", round(clean_bundle_zip_path.stat().st_size / (1024 * 1024), 2), "MB")

# -----------------------------
# 10. Create full package ZIP in Google Drive
# -----------------------------
full_zip_path = dataset_root / "industrial_park_load_FULL_PACKAGE.zip"

with zipfile.ZipFile(full_zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    zf.write(download_path, arcname="raw/" + download_path.name)

    for file in extracted_dir.rglob("*"):
        if file.is_file():
            rel = str(file.relative_to(extracted_dir))
            if "__MACOSX" in rel or file.name.startswith("._"):
                continue
            zf.write(file, arcname="extracted/" + rel)

    zf.write(clean_bundle_zip_path, arcname="bundle/" + clean_bundle_zip_path.name)
    zf.write(manifest_path, arcname="manifest/" + manifest_path.name)

print("Created full package ZIP in Google Drive:", full_zip_path)
print("Full package size:", round(full_zip_path.stat().st_size / (1024 * 1024), 2), "MB")

# -----------------------------
# 11. Copy final ZIP to /content for reliable PC download
# -----------------------------
runtime_copy = Path("/content/industrial_park_load_FULL_PACKAGE.zip")
shutil.copy2(full_zip_path, runtime_copy)

print("Runtime backup created:", runtime_copy)
print("Runtime backup size:", round(runtime_copy.stat().st_size / (1024 * 1024), 2), "MB")

# -----------------------------
# 12. Final summary
# -----------------------------
print("\n===== SUMMARY =====")
print("Dataset root         :", dataset_root)
print("Raw ZIP              :", download_path)
print("Extracted folder     :", extracted_dir)
print("Manifest             :", manifest_path)
print("Clean bundle ZIP     :", clean_bundle_zip_path)
print("Full package ZIP     :", full_zip_path)
print("Runtime backup ZIP   :", runtime_copy)
print("===================")

# -----------------------------
# 13. Download final package to PC
# -----------------------------
print("\nStarting browser download...")
files.download(str(runtime_copy))

Dataset Name : Industrial Park Electric Power Load Dataset
Source Page  : https://github.com/Industrialpark/SEMLab_HFUT-Building-Electricpowerloaddata
Download URL : https://codeload.github.com/Industrialpark/SEMLab_HFUT-Building-Electricpowerloaddata/zip/refs/heads/main
Mounted at /content/drive

Created folders:
 - /content/drive/MyDrive/PhD_Thesis_Datasets
 - /content/drive/MyDrive/PhD_Thesis_Datasets/10_industrial_park_load
 - /content/drive/MyDrive/PhD_Thesis_Datasets/10_industrial_park_load/raw
 - /content/drive/MyDrive/PhD_Thesis_Datasets/10_industrial_park_load/extracted
 - /content/drive/MyDrive/PhD_Thesis_Datasets/10_industrial_park_load/bundle

Download complete: /content/drive/MyDrive/PhD_Thesis_Datasets/10_industrial_park_load/raw/industrial_park_load_main.zip
ZIP size: 1.02 MB

Extracting ZIP...
Extraction complete: /content/drive/MyDrive/PhD_Thesis_Datasets/10_industrial_park_load/extracted

Total extracted files: 24
Usable files: 13
 - /content/drive/MyDrive/PhD_Thesis_

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>